In [1]:
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import pandas as pd

# 1. Загрузка данных
penguins = sns.load_dataset('penguins').dropna()

# 2. Разделение на X и y
X = penguins.drop(columns=['species'])
y = penguins['species']

# 3. One-Hot Encoding для признаков (island, sex)
X = pd.get_dummies(X, drop_first=True, dtype=float)

# 4. Кодирование таргета в числа (0, 1, 2)
y = y.astype('category').cat.codes

# 5. Масштабирование (обязательно для нейросетей!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 7. Превращение в тензоры PyTorch
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.long) # long - это int64, нужен для классов
y_test_t = torch.tensor(y_test.values, dtype=torch.long)

In [2]:
import torch.nn as nn

class PenguinNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        # Скрытый слой (принимает все признаки, выдает 16 нейронов)
        self.fc1 = nn.Linear(input_size, 16)
        # Функция активации
        self.relu = nn.ReLU()
        # Выходной слой (выдает вероятности для 3 классов)
        self.fc2 = nn.Linear(16, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# Размерность входа = количество колонок в X
input_size = X_train_t.shape[1] 
num_classes = 3 # Adelie, Chinstrap, Gentoo

model = PenguinNet(input_size, num_classes)

In [3]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [4]:
epochs = 100
for epoch in range(epochs):
    # 1. Forward
    y_logits = model(X_train_t)
    
    # 2. Loss
    loss = criterion(y_logits, y_train_t)
    
    # 3. Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Эпоха {epoch+1}, Loss: {loss.item():.4f}")

Эпоха 20, Loss: 0.3420
Эпоха 40, Loss: 0.0716
Эпоха 60, Loss: 0.0292
Эпоха 80, Loss: 0.0187
Эпоха 100, Loss: 0.0145


In [5]:
from sklearn.metrics import accuracy_score

# Отключаем градиенты для теста (экономит память)
with torch.no_grad():
    test_logits = model(X_test_t)
    # argmax(1) находит индекс максимального логита в каждой строке
    test_preds = torch.argmax(test_logits, dim=1) 

# Превращаем тензоры обратно в numpy для sklearn
test_preds_np = test_preds.numpy()
y_test_np = y_test_t.numpy()

print(f"Accuracy: {accuracy_score(y_test_np, test_preds_np):.4f}")

Accuracy: 1.0000


In [6]:
import torch.nn as nn

class PenguinNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        # Скрытый слой (принимает все признаки, выдает 16 нейронов)
        self.fc1 = nn.Linear(input_size, 16)
        # Функция активации
        self.sigmoid = nn.Sigmoid()
        # Выходной слой (выдает вероятности для 3 классов)
        self.fc2 = nn.Linear(16, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.sigmoid(out)
        out = self.fc2(out)
        return out

# Размерность входа = количество колонок в X
input_size = X_train_t.shape[1] 
num_classes = 3 # Adelie, Chinstrap, Gentoo

model = PenguinNet(input_size, num_classes)

In [7]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [8]:
epochs = 100
for epoch in range(epochs):
    # 1. Forward
    y_logits = model(X_train_t)
    
    # 2. Loss
    loss = criterion(y_logits, y_train_t)
    
    # 3. Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Эпоха {epoch+1}, Loss: {loss.item():.4f}")

Эпоха 20, Loss: 0.5995
Эпоха 40, Loss: 0.2804
Эпоха 60, Loss: 0.1364
Эпоха 80, Loss: 0.0763
Эпоха 100, Loss: 0.0502


In [9]:
from sklearn.metrics import accuracy_score

# Отключаем градиенты для теста (экономит память)
with torch.no_grad():
    test_logits = model(X_test_t)
    # argmax(1) находит индекс максимального логита в каждой строке
    test_preds = torch.argmax(test_logits, dim=1) 

# Превращаем тензоры обратно в numpy для sklearn
test_preds_np = test_preds.numpy()
y_test_np = y_test_t.numpy()

print(f"Accuracy: {accuracy_score(y_test_np, test_preds_np):.4f}")

Accuracy: 1.0000


In [10]:
import torch.nn as nn

class PenguinNet(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        # Скрытый слой (принимает все признаки, выдает 16 нейронов)
        self.fc1 = nn.Linear(input_size, 16)
        # Функция активации
        # Выходной слой (выдает вероятности для 3 классов)
        self.fc2 = nn.Linear(16, num_classes)

    def forward(self, x):
        out = self.fc1(x)
        out = self.fc2(out)
        return out

# Размерность входа = количество колонок в X
input_size = X_train_t.shape[1] 
num_classes = 3 # Adelie, Chinstrap, Gentoo

model = PenguinNet(input_size, num_classes)

In [11]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [12]:
epochs = 100
for epoch in range(epochs):
    # 1. Forward
    y_logits = model(X_train_t)
    
    # 2. Loss
    loss = criterion(y_logits, y_train_t)
    
    # 3. Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f"Эпоха {epoch+1}, Loss: {loss.item():.4f}")

Эпоха 20, Loss: 0.1388
Эпоха 40, Loss: 0.0379
Эпоха 60, Loss: 0.0229
Эпоха 80, Loss: 0.0178
Эпоха 100, Loss: 0.0149


In [13]:
from sklearn.metrics import accuracy_score

# Отключаем градиенты для теста (экономит память)
with torch.no_grad():
    test_logits = model(X_test_t)
    # argmax(1) находит индекс максимального логита в каждой строке
    test_preds = torch.argmax(test_logits, dim=1) 

# Превращаем тензоры обратно в numpy для sklearn
test_preds_np = test_preds.numpy()
y_test_np = y_test_t.numpy()

print(f"Accuracy: {accuracy_score(y_test_np, test_preds_np):.4f}")

Accuracy: 1.0000
